# LLM Playground — AlphaForge Anton

Test every provider, streaming, gateway routing, and the live concierge endpoint.

Notebook lives at `concierge/notebooks/`. The LLM package is at `concierge/llm/src/`.

In [ ]:
import asyncio, os, sys
from pathlib import Path
from dotenv import load_dotenv

# concierge/notebooks/ → concierge/ → project root
CONCIERGE = Path.cwd().parents[0]
ROOT = Path.cwd().parents[1]
LLM_SRC = CONCIERGE / "llm" / "src"

if str(LLM_SRC) not in sys.path:
    sys.path.insert(0, str(LLM_SRC))

# Load credentials — last file wins
for _f in [".env", ".env.local", ".env.cred.local"]:
    _p = ROOT / _f
    if _p.exists():
        load_dotenv(_p, override=True)
        print(f"Loaded {_f}")

from alphaforge_anton_llm import (
    REGISTRY, Message, QueryType, create_gateway, CostGuardError,
)
from alphaforge_anton_llm.router import QueryRouter

def msg(role: str, content: str) -> Message:
    return Message(role=role, content=content)

QUERY = [msg("user", "What is the current trend for Nifty 50?")]

print(f"\nalphaforge_anton_llm loaded OK")
print(f"Registered providers: {list(REGISTRY)}")
print(f"LLM src:              {LLM_SRC}")

## 1. Health check — all providers

In [ ]:
gw = create_gateway()
healths = await gw.health()

print(f"{'Provider':<16} {'OK?':<6} {'Error'}")
print("-" * 65)
for name, h in sorted(healths.items()):
    avail = "✓" if h.available else "✗"
    err = (h.last_error or "")[:45]
    print(f"  {name:<14} {avail:<6} {err}")

available = [n for n, h in healths.items() if h.available]
print(f"\nAvailable ({len(available)}): {available}")

## 2. All providers — completion sweep

Runs every available provider against the same query. Skips unavailable ones.

In [ ]:
from alphaforge_anton_llm.providers import (
    GeminiAdapter, GroqAdapter, CerebrasAdapter,
    MistralAdapter, OpenRouterAdapter, HuggingFaceAdapter,
)

ADAPTERS = [
    ("gemini",      GeminiAdapter()),
    ("groq",        GroqAdapter()),
    ("cerebras",    CerebrasAdapter()),
    ("mistral",     MistralAdapter()),
    ("openrouter",  OpenRouterAdapter()),
    ("huggingface", HuggingFaceAdapter()),
]

for name, adapter in ADAPTERS:
    h = await adapter.health()
    status = "✓" if h.available else f"✗ {(h.last_error or '')[:40]}"
    if not h.available:
        print(f"  {name:<14} {status}")
        continue
    try:
        r = await adapter.complete(QUERY)
        print(f"  {name:<14} ✓  {r.model:<35} {r.prompt_tokens}+{r.completion_tokens} tok")
        print(f"    {r.content[:120].strip()}...")
    except Exception as e:
        print(f"  {name:<14} ERROR: {e}")
    print()

## 3. Streaming — token-by-token output

Tests `.astream()` on providers that support it.

In [ ]:
STREAM_PROVIDERS = [
    ("groq",       GroqAdapter()),
    ("cerebras",   CerebrasAdapter()),
    ("mistral",    MistralAdapter()),
    ("openrouter", OpenRouterAdapter()),
]

for name, adapter in STREAM_PROVIDERS:
    h = await adapter.health()
    if not h.available:
        print(f"  {name}: unavailable, skip")
        continue
    if not (getattr(adapter, "supports_streaming", False) and hasattr(adapter, "astream")):
        print(f"  {name}: no streaming support")
        continue
    print(f"  {name} → streaming: ", end="", flush=True)
    chunks = 0
    last = None
    try:
        async for snap in adapter.astream(QUERY):
            chunks += 1
            last = snap
        print(f"{chunks} chunks  {last.prompt_tokens}+{last.completion_tokens} tok  ✓")
    except Exception as e:
        print(f"ERROR: {e}")

## 4. CostGuard — Claude blocked without confirmation

In [ ]:
from alphaforge_anton_llm.cost_guard import CostGuard, CostGuardError

guard = CostGuard()

# Should raise
try:
    guard.check("claude-sdk", confirmed=False)
    print("ERROR: should have raised CostGuardError")
except CostGuardError as e:
    print(f"✓ Blocked without confirmation: {e}")

# Should pass
try:
    guard.check("claude-sdk", confirmed=True)
    print("✓ Passes with confirmed=True")
except CostGuardError:
    print("ERROR: should have passed with confirmed=True")

# Free providers pass unconditionally
guard.check("gemini", confirmed=False)
print("✓ Free provider (gemini) passes without confirmation")

## 5. Gateway — auto-routing by QueryType

In [ ]:
gw = create_gateway()

print(f"{'QueryType':<24} {'Provider':<14} {'Model'}")
print("-" * 70)
for qt in QueryType:
    try:
        r = await gw.complete(QUERY, query_type=qt)
        print(f"  {qt.value:<22} {r.provider:<14} {r.model}")
    except CostGuardError as e:
        print(f"  {qt.value:<22} COST GUARD: {e}")
    except Exception as e:
        print(f"  {qt.value:<22} ERROR: {e}")

## 6. Gateway — SSE streaming via `gw.stream()`

In [ ]:
gw = create_gateway()

print("Streaming FACTOID query...")
final = None
async for snap in gw.stream(QUERY, query_type=QueryType.FACTOID):
    final = snap
if final:
    print(f"Provider: {final.provider}  Model: {final.model}")
    print(f"Tokens: {final.prompt_tokens}+{final.completion_tokens}")
    print(f"\n{final.content[:300]}")

## 7. Router — chain inspection

In [ ]:
router = QueryRouter()
print(f"{'QueryType':<24} {'Chain'}")
print("-" * 60)
for qt in QueryType:
    chain = router.chain_for(qt)
    print(f"  {qt.value:<22} {chain}")

## 8. Rate limiter state

In [ ]:
from alphaforge_anton_llm.rate_limiter import RateLimiter

rl = RateLimiter()
print(f"{'Provider':<16} {'Remaining'}")
print("-" * 35)
for name in REGISTRY:
    rem = rl.remaining(name)
    label = f"{rem:.1f} tokens" if rem is not None else "unlimited"
    print(f"  {name:<14} {label}")

## 9. Live concierge endpoint — httpx SSE

Requires the backend to be running on port 8000. Reads `af_token` from env or prompts.

Set `AF_TOKEN` in `.env.cred.local` or run `backend` first to get a token.

In [ ]:
import httpx, json

BACKEND = "http://localhost:8000"
TOKEN = os.environ.get("AF_TOKEN", "")

if not TOKEN:
    print("⚠ AF_TOKEN not set — set it in .env.cred.local to test authenticated endpoints")
else:
    payload = {
        "messages": [{"role": "user", "content": "What is my current portfolio allocation?"}],
        "provider": "auto",
        "auto_level": "top",
    }
    headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

    print("POST /api/v1/concierge — streaming...\n")
    chunks_received = 0
    async with httpx.AsyncClient(timeout=30) as client:
        async with client.stream(
            "POST", f"{BACKEND}/api/v1/concierge",
            headers=headers, json=payload,
        ) as resp:
            print(f"HTTP {resp.status_code}")
            if resp.status_code != 200:
                print(await resp.aread())
            else:
                async for line in resp.aiter_lines():
                    if not line.startswith("data: "):
                        continue
                    raw = line[6:].strip()
                    if raw == "[DONE]":
                        print(f"\n\n[DONE] — {chunks_received} SSE frames received")
                        break
                    try:
                        frame = json.loads(raw)
                        if "error" in frame:
                            print(f"ERROR: {frame['error']}")
                            break
                        if "content" in frame:
                            chunks_received += 1
                            print(frame["content"], end="", flush=True)
                    except json.JSONDecodeError:
                        pass

## 10. Claude SDK — direct call (requires confirmation)

Only runs if `CLAUDE_ENABLED=1` is set to prevent accidental billing.

In [ ]:
from alphaforge_anton_llm.providers.claude_sdk import ClaudeSdkAdapter

if os.environ.get("CLAUDE_ENABLED") != "1":
    print("Skipped — set CLAUDE_ENABLED=1 in env to run this cell")
else:
    claude = ClaudeSdkAdapter()
    h = await claude.health()
    print("Health:", h)
    if h.available:
        r = await claude.complete(QUERY)  # ClaudeSdkAdapter bypasses CostGuard
        print(f"Model: {r.model}  tokens: {r.prompt_tokens}+{r.completion_tokens}")
        print(r.content[:300])